<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Mini_projet_W4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prédiction des maladies cardiaques par régression logistique

Ce notebook a pour objectif de développer un modèle de régression logistique pour prédire la présence de maladies cardiaques à partir d'un ensemble de données UCI. Il couvre l'analyse exploratoire des données (EDA), le prétraitement des données, l'entraînement et l'évaluation du modèle.

## 1. Importation des bibliothèques nécessaires

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import zipfile
import os

## 2. Préparation des données

Nous allons commencer par décompresser le fichier de données fourni et charger le jeu de données.

In [ ]:
# Décompresser le fichier zip
zip_file_path = '/content/UCI Heart Disease Data.zip'
extract_dir = '/content/heart_disease_data'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Lister les fichiers extraits pour trouver le CSV
print("Fichiers extraits:")
for root, dirs, files in os.walk(extract_dir):
    for name in files:
        print(os.path.join(root, name))

# Assumons que le fichier CSV se trouve directement dans le dossier extrait et s'appelle 'heart.csv'
# Si le nom du fichier est différent, ajustez cette ligne.
csv_file_path = os.path.join(extract_dir, 'heart.csv') # Common name, adjust if needed

# Charger les données
try:
    df = pd.read_csv(csv_file_path)
    print("\nDonnées chargées avec succès.")
except FileNotFoundError:
    print(f"Erreur: Le fichier CSV '{csv_file_path}' n'a pas été trouvé. Veuillez vérifier le nom du fichier dans le dossier extrait.")
    # Fallback to look for a different common name if heart.csv is not found
    csv_file_path = os.path.join(extract_dir, 'framingham.csv') # Another common name
    try:
        df = pd.read_csv(csv_file_path)
        print("\nDonnées chargées avec succès en utilisant 'framingham.csv'.")
    except FileNotFoundError:
        print("Erreur: Aucun des noms de fichiers CSV courants ('heart.csv', 'framingham.csv') n'a été trouvé.")
        print("Veuillez vérifier manuellement le contenu du dossier extrait et ajuster 'csv_file_path'.")

## 3. Analyse Exploratoire des Données (EDA)

Nous allons explorer le jeu de données pour comprendre sa structure, la distribution des variables et identifier les valeurs manquantes.

In [ ]:
# Afficher les premières lignes du DataFrame
print("Premières 5 lignes du DataFrame:")
display(df.head())

# Afficher des informations générales sur le DataFrame
print("\nInformations sur le DataFrame:")
display(df.info())

# Afficher les statistiques descriptives
print("\nStatistiques descriptives:")
display(df.describe())

# Vérifier les valeurs manquantes
print("\nNombre de valeurs manquantes par colonne:")
display(df.isnull().sum())

### Visualisation des distributions

In [ ]:
# Distribution des variables numériques
plt.figure(figsize=(15, 10))
for i, column in enumerate(df.select_dtypes(include=np.number).columns):
    if column != 'target': # Exclure la variable cible pour les histogrammes standards
        plt.subplot(3, 4, i + 1)
        sns.histplot(df[column].dropna(), kde=True)
        plt.title(f'Distribution de {column}')
plt.tight_layout()
plt.show()

# Distribution de la variable cible
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df)
plt.title('Distribution de la variable cible (Présence de maladie cardiaque)')
plt.xlabel('Maladie cardiaque (0 = Non, 1 = Oui)')
plt.ylabel('Nombre de cas')
plt.show()

### Corrélation entre les variables

In [ ]:
# Matrice de corrélation
plt.figure(figsize=(12, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matrice de corrélation des caractéristiques')
plt.show()

## 4. Prétraitement des données

Nous allons gérer les valeurs manquantes, encoder les variables catégorielles et mettre à l'échelle les caractéristiques numériques.

In [ ]:
# Gérer les valeurs manquantes (imputation simple par la médiane pour les numériques)
# Pour cet ensemble de données, il peut y avoir des valeurs manquantes représentées par 9999 ou d'autres marqueurs.
# Basé sur l'EDA, si des valeurs manquantes sont identifiées, nous les traiterons ici.

# Exemple: Si 'ca' ou 'thal' ont des valeurs manquantes, nous pouvons les imputer ou les supprimer.
# Pour cet exemple, je vais imputer les valeurs manquantes avec la médiane pour les colonnes numériques.

for col in df.select_dtypes(include=np.number).columns:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Valeurs manquantes dans '{col}' imputées avec la médiane: {median_val}")

# Vérifier à nouveau les valeurs manquantes après imputation
print("\nNombre de valeurs manquantes après imputation:")
display(df.isnull().sum())

In [ ]:
# Séparer les caractéristiques (X) et la variable cible (y)
X = df.drop('target', axis=1)
y = df['target']

# Identifier les colonnes numériques et catégorielles
# Basé sur la documentation UCI Heart Disease, les colonnes catégorielles sont:
# sex, cp, fbs, restecg, exang, slope, ca, thal
# Les colonnes numériques sont: age, trestbps, chol, oldpeak, thalach

# Si les colonnes 'ca' et 'thal' ont été traitées comme numériques (comme après imputation), il faut les inclure dans les catégorielles si elles sont ordinales ou nominales.
# Vérifions les valeurs uniques pour ces colonnes pour décider.

# Exemple de colonnes catégorielles (à adapter en fonction de l'EDA et de la documentation du dataset)
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
numerical_features = X.select_dtypes(include=np.number).columns.drop(categorical_features, errors='ignore').tolist()

print(f"Colonnes numériques: {numerical_features}")
print(f"Colonnes catégorielles: {categorical_features}")

# Créer un préprocesseur pour appliquer différentes transformations à différents types de colonnes
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

## 5. Division des données en ensembles d'entraînement et de test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Taille de l'ensemble d'entraînement (X_train): {X_train.shape}")
print(f"Taille de l'ensemble de test (X_test): {X_test.shape}")
print(f"Distribution de la cible dans l'entraînement:\n{y_train.value_counts(normalize=True)}")
print(f"Distribution de la cible dans le test:\n{y_test.value_counts(normalize=True)}")

## 6. Entraînement du modèle de régression logistique

Nous allons construire un pipeline qui inclut le prétraitement des données et le modèle de régression logistique.

In [ ]:
# Créer un pipeline de prétraitement et de modélisation
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, solver='liblinear')) # 'liblinear' est bon pour les petits datasets
])

# Entraîner le modèle
model_pipeline.fit(X_train, y_train)

print("Modèle de régression logistique entraîné avec succès.")

## 7. Évaluation du modèle

Nous allons évaluer les performances du modèle sur l'ensemble de test en utilisant diverses métriques et une matrice de confusion.

In [ ]:
# Faire des prédictions sur l'ensemble de test
y_pred = model_pipeline.predict(X_test)

# Calculer les métriques d'évaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Exactitude (Accuracy): {accuracy:.4f}")
print(f"Précision (Precision): {precision:.4f}")
print(f"Rappel (Recall): {recall:.4f}")
print(f"Score F1 (F1-Score): {f1:.4f}")

### Matrice de confusion

In [ ]:
# Matrice de confusion
conf_matrix = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Prédit Non-maladie (0)', 'Prédit Maladie (1)'],
            yticklabels=['Réel Non-maladie (0)', 'Réel Maladie (1)'])
plt.xlabel('Prédiction')
plt.ylabel('Valeur réelle')
plt.title('Matrice de Confusion')
plt.show()

### Rapport de classification

In [ ]:
# Rapport de classification
print("Rapport de Classification:\n")
print(classification_report(y_test, y_pred))

## 8. Conclusion

Ce notebook a démontré le processus complet de prédiction de maladies cardiaques à l'aide d'un modèle de régression logistique. Nous avons couvert le chargement des données, l'EDA, le prétraitement des données (gestion des valeurs manquantes, encodage des catégories et mise à l'échelle), l'entraînement du modèle et son évaluation. Les performances du modèle sont résumées par l'exactitude, la précision, le rappel, le score F1, la matrice de confusion et le rapport de classification.